[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/notebooks/v030/v0302_single_neuron_parameter_sweep.ipynb)

# v0.3.2: Single Neuron Parameter Sweep Tutorial

**Tutorial ID:** v0302_single_neuron_parameter_sweep  
**jaxfne version:** 0.2.30  
**Truth status:** `truth_safe_unverified`

---

## Section 1: Learning Objectives

After completing this tutorial, you will be able to:

1. Use `jtfne.with_emitter_parameters()` to vary Izhikevich `a` and `d` parameters on an existing model.
2. Run a deterministic grid sweep and collect proxy firing rates.
3. Visualise regime transitions as a heatmap and line plot.
4. Verify claim gates are preserved across all sweep points.
5. Distinguish computational parameter sensitivity from biological validation.

## Section 2: Biological/Computational Question

**Question:** How do `a` (recovery time scale) and `d` (after-spike reset) affect proxy firing rate?

**Computational framing:** Grid sweep over `a` and `d` using `jtfne.with_emitter_parameters`. Firing rates are proxy outputs — not calibrated to biological neural subtypes.

**What this does NOT claim:** That any sweep point corresponds to a named neuron type (RS, FS, IB, chattering).

## Section 3: Mathematical Glossary

$$\frac{dv}{dt} = 0.04v^2 + 5v + 140 - u + I(t)$$

$$\frac{du}{dt} = a(bv - u)$$

**Reset:** when $v \geq 30$ mV: $v \leftarrow c$, $u \leftarrow u + d$

**Term Glossary:**
- $a$ — recovery time scale (ms⁻¹, proxy): larger → faster recovery → higher proxy rate
- $d$ — after-spike reset (proxy): larger → stronger adaptation → lower proxy rate
- $v$ — membrane voltage (mV, model units)
- $u$ — recovery variable (dimensionless)

**Implementation:** `jtfne.with_emitter_parameters(model, a=..., d=...)`

**Claim boundary:** Mathematical model dynamics; not validated against patch-clamp data.

## Section 4: Canonical Import

In [ ]:
# Install jaxfne (Colab). Uncomment if needed:
# !pip install jaxfne==0.2.30

import jaxfne as jtfne
import jax.numpy as jnp
import numpy as np
import json
import matplotlib.pyplot as plt
print(f"jaxfne version: {jtfne.__version__}")

## Section 5: Configuration and Base Model

In [ ]:
run = jtfne.runtime(device_type="auto", dtype="float32", x64_enabled=False, seed=42)

cfg = (
    jtfne.configuration()
    .network(name="v030_02_parameter_sweep", kind="isolated_neuron", n=1, cell_types={"E": 1.0})
    .emitter(family="izhikevich", preset="cortical_eig")
    .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann", gauge="mean_zero")
    .probe(name="single_channel_16contact", modes=["spikes", "V_m", "source", "LFP", "CSD"], n_contacts=16)
)

base_model = jtfne.construct(cfg)

# Confirm baseline parameters (cortical_eig preset)
e = base_model.params["emitter"]
print(f"Baseline a={float(e.a[0]):.3f}, b={float(e.b[0]):.3f}, c={float(e.c[0]):.1f}, d={float(e.d[0]):.1f}, drive={float(e.drive[0]):.1f}")

## Section 6: Parameter Sweep

In [ ]:
DURATION_MS = 500.0
DT_MS = 0.1
SEED = 42

sim_spec = jtfne.simulation(duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED, runtime=run)

A_VALUES = [0.02, 0.05, 0.10]
D_VALUES = [2.0, 6.0, 8.0, 12.0]

sweep_results = []
firing_grid = np.zeros((len(A_VALUES), len(D_VALUES)))

for i, a_val in enumerate(A_VALUES):
    for j, d_val in enumerate(D_VALUES):
        model = jtfne.with_emitter_parameters(base_model, a=a_val, d=d_val)
        signals = model.simulate(sim_spec)
        spikes = np.array(signals.spikes)
        n_spikes = len(np.where(spikes[:, 0] > 0.5)[0])
        hz = float((n_spikes / DURATION_MS) * 1000.0)
        firing_grid[i, j] = hz
        sweep_results.append({"a": a_val, "d": d_val, "firing_rate_hz": hz, "n_spikes": n_spikes})
        print(f"  a={a_val:.2f}, d={d_val:.1f} → {hz:.1f} Hz")

print(f"\nFiring rate range: [{firing_grid.min():.1f}, {firing_grid.max():.1f}] Hz")

## Section 7: Probe Readout (baseline point)

In [ ]:
# Representative readout at baseline (a=0.02, d=8 = cortical_eig default)
baseline_model = jtfne.with_emitter_parameters(base_model, a=0.02, d=8.0)
baseline_signals = baseline_model.simulate(sim_spec)

V_m = np.array(baseline_signals.V_m)
spikes = np.array(baseline_signals.spikes)
baseline_spikes = len(np.where(spikes[:, 0] > 0.5)[0])
baseline_rate = float((baseline_spikes / DURATION_MS) * 1000.0)

print(f"Baseline V_m shape:    {baseline_signals.V_m.shape}")
print(f"Baseline spikes shape: {baseline_signals.spikes.shape}")
print(f"Baseline firing rate:  {baseline_rate:.1f} Hz")
print(f"Baseline gate (2-25 Hz): {'PASS' if 2.0 <= baseline_rate <= 25.0 else 'FAIL'}")
print(f"All V_m finite:        {bool(np.all(np.isfinite(V_m)))}")

## Section 8: Manifest and Claim Gates

In [ ]:
manifest = {
    "tutorial_id": "v0302_single_neuron_parameter_sweep",
    "jaxfne_version": jtfne.__version__,
    "basis": {
        "truth_mode": "truth_safe_unverified",
        "claim_level": "computational_scaffold",
        "field_solver_status": "laminar_proxy_no_pde",
        "physical_amplitude_claim_allowed": False,
        "biological_metabolism_claim_allowed": False,
    },
    "sweep_summary": {
        "n_points": len(sweep_results),
        "rate_min_hz": float(firing_grid.min()),
        "rate_max_hz": float(firing_grid.max()),
        "baseline_rate_hz": baseline_rate,
        "baseline_gate_2_25_hz": 2.0 <= baseline_rate <= 25.0,
    },
}

print("truth_mode          :", manifest["basis"]["truth_mode"])
print("claim_level         :", manifest["basis"]["claim_level"])
print("field_solver_status :", manifest["basis"]["field_solver_status"])
print("physical_amplitude  :", manifest["basis"]["physical_amplitude_claim_allowed"])
print("baseline_rate_hz    :", manifest["sweep_summary"]["baseline_rate_hz"])
print("baseline_gate       :", manifest["sweep_summary"]["baseline_gate_2_25_hz"])

json.dumps(manifest, allow_nan=False)
print("manifest: JSON-safe ✓")

## Section 9: Figures

Figures are generated by `examples/v032_single_neuron_parameter_sweep.py` and stored in
`docs/tutorials_v030/_static/figures/`. See the docs page for the committed figures.

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(firing_grid, aspect="auto", origin="upper", cmap="viridis", vmin=0, vmax=max(firing_grid.max(), 25.0))
ax.set_xticks(range(len(D_VALUES))); ax.set_xticklabels([f"{d:.1f}" for d in D_VALUES])
ax.set_yticks(range(len(A_VALUES))); ax.set_yticklabels([f"{a:.2f}" for a in A_VALUES])
ax.set_xlabel("d — after-spike reset (proxy units)")
ax.set_ylabel("a — recovery time scale (proxy units)")
ax.set_title("v0.3.2: Izhikevich Sweep — Firing Rate (Hz)\n(Proxy readout, not biological validation)")
plt.colorbar(im, ax=ax, label="Firing rate (Hz)")
for i in range(len(A_VALUES)):
    for j in range(len(D_VALUES)):
        ax.text(j, i, f"{firing_grid[i,j]:.1f}", ha="center", va="center", fontsize=9, color="white")
plt.tight_layout(); plt.show()

In [ ]:
# Regime line plot
fig, ax = plt.subplots(figsize=(9, 4))
for i, a_val in enumerate(A_VALUES):
    ax.plot(D_VALUES, firing_grid[i, :], marker="o", label=f"a={a_val:.2f}")
ax.axhline(2.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7, label="Gate low (2 Hz)")
ax.axhline(25.0, color="gray", linestyle=":", linewidth=0.8, alpha=0.7, label="Gate high (25 Hz)")
ax.set_xlabel("d — after-spike reset (proxy units)")
ax.set_ylabel("Firing rate (Hz)")
ax.set_title("v0.3.2: Firing Rate vs. d (per a)\n(Proxy readout, not biological validation)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Section 10: Interpretation

The sweep shows proxy firing rate increases with `a` and decreases with `d`, consistent with the Izhikevich model's mathematical structure. Larger `a` accelerates the recovery variable, shortening the effective refractory period. Larger `d` increases the post-spike recovery increment, strengthening adaptation. The baseline `cortical_eig` point (a=0.02, d=8) produces 12 Hz — inside the 2–25 Hz acceptance gate. These trends are deterministic model outputs, not fitted to neural data.

## Section 11: Failure Modes

| Failure Mode | Symptom | Mitigation |
|-------------|---------|------------|
| All-zero firing at large `d` | rate = 0 Hz, gate FAIL | Reduce `d`; check drive level |
| Rate explosion at large `a` | rate ≫ 25 Hz | Reduce `a` or `drive_scale` |
| NaN in manifest | `json.dumps` raises ValueError | Check finite signals |
| Baseline gate FAIL | collector rejects scenario | Verify `a=0.02, d=8` with `cortical_eig` |

## Section 12: Exercises

1. Extend the grid to `a` in `[0.02, 0.04, 0.06, 0.08, 0.10]` (5 values). How does the heatmap change?
2. Add `drive_scale` as a third axis. Try `drive_scale` in `[0.8, 1.0, 1.5]`.
3. Add `c` override (`c=-65` vs `c=-55`). Does the firing pattern change?
4. Set `seed=1` and re-run. Verify determinism across seeds.
5. Increase `duration_ms` to 1000 ms. Does the rate estimate stabilise?

## Section 13: What This Tutorial Does NOT Claim

**This tutorial is a computational scaffold.**

This tutorial does **not** claim:

- The parameter grid corresponds to named biological neuron subtypes (RS, FS, IB, chattering)
- `a` and `d` values are calibrated against electrophysiology recordings
- Firing rate changes reflect real neural adaptation or accommodation
- The proxy LFP/CSD amplitudes correspond to physical microvolts
- Any regime boundary maps to a validated biological transition
- The dynamics have been validated against any empirical dataset

**To make physical-amplitude, biological, or mechanistic claims, you must supply:**
- Calibration evidence (empirical recordings with known units)
- A solved field (Poisson or Maxwell equations with physical conductivity)
- A peer-reviewed validation protocol

Until these are supplied, all outputs remain computational scaffolds with `physical_amplitude_claim_allowed: False`.

---

**End of Tutorial**